# CPDS-AI: Audio Classification (Multi-Dataset Training)

This notebook trains a lightweight AudioCNN to classify **baby cry vs environmental noise** by merging multiple cry datasets for maximum generalization.

### Data Sources
| # | Dataset | Source | Approx. Files |
|---|---------|--------|---------------|
| 1 | Baby Crying Sounds (Menna Ahmed) | Kaggle | ~1313 |
| 2 | Donate-a-cry Corpus | GitHub | ~1585 |
| 3 | ESC-50 `crying_baby` class | GitHub | 40 |
| 4 | ESC-50 (vehicle/environment noise) | GitHub | ~640 |

### Pipeline
1. Download & merge all cry sources
2. Filter with MIT AST pseudo-labeling (discard non-cry samples)
3. Extract loudest 2-second window + peak normalize
4. Train AudioCNN with SpecAugment & Early Stopping
5. Evaluate on held-out test set
6. Export ONNX (float32)

In [ ]:
!pip install -q librosa onnx onnxruntime==1.20.1 soundfile onnxscript transformers

In [ ]:
import librosa
import torch
import torch.nn as nn
import json
import csv
import shutil
import random
import subprocess
import numpy as np
import soundfile as sf
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

## 0. Configuration

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

SR = 16000
DURATION = 2.0
N_FFT = 2048
N_MELS = 128
N_FRAMES = 63
MAX_SAMPLES_PER_CLASS = 2000
EVAL_HOLDOUT_PER_CLASS = 50
BATCH_SIZE = 32
NUM_EPOCHS = 100
LEARNING_RATE = 1e-3
PATIENCE = 7
EARLY_STOP_MIN_EPOCHS = 10

AUDIO_EXTENSIONS = {".wav"}
SOURCE_AUDIO_EXTENSIONS = {".wav", ".mp3", ".ogg", ".flac", ".m4a", ".caf", ".3gp"}

VEHICLE_NOISE_CATEGORIES = {
    "engine", "car_horn", "siren", "rain", "wind", "door_wood_knock",
    "dog", "cat", "laughing", "breathing", "coughing", "sneezing", "snoring",
    "footsteps", "chirping_birds", "thunderstorm",
}
CLASS_NAMES = ["noise", "cry"]

## 1. Audio Preprocessing Utilities

In [ ]:
def select_loudest_window(y, sr, duration=2.0, hop_fraction=0.1):
    """Return the `duration`-second window with highest RMS energy (pad short clips)."""
    target_len = int(sr * duration)
    if len(y) <= target_len:
        return np.pad(y, (0, target_len - len(y)))
    hop = max(1, int(target_len * hop_fraction))
    best_start, best_energy = 0, -1.0
    for start in range(0, len(y) - target_len + 1, hop):
        segment = y[start:start + target_len]
        energy = float(np.sum(segment.astype(np.float64) ** 2))
        if energy > best_energy:
            best_energy, best_start = energy, start
    return y[best_start:best_start + target_len]


def _peak_normalize(y):
    y = np.asarray(y, dtype=np.float32)
    max_val = np.max(np.abs(y))
    if np.isfinite(max_val) and max_val > 1e-4:
        y = (y / max_val) * 0.95
    return y


def split_holdout_and_train_pool(population, seed, holdout_size):
    """Deterministic shuffle -> (eval_holdout, train_pool), disjoint by construction."""
    shuffled = list(population)
    random.Random(seed).shuffle(shuffled)
    return shuffled[:holdout_size], shuffled[holdout_size:]


def convert_to_wav(source, destination, prefix, sr=16000, duration=2.0):
    """Load any audio file, crop loudest window, normalize, save as 16kHz WAV."""
    destination_path = destination / f"{prefix}_{source.stem}.wav"
    suffix = 1
    while destination_path.exists():
        destination_path = destination / f"{prefix}_{suffix}_{source.stem}.wav"
        suffix += 1
    try:
        y, loaded_sr = librosa.load(str(source), sr=sr, mono=True)
        if y.size == 0:
            y = np.zeros(int(sr * duration), dtype=np.float32)
        y = select_loudest_window(y, loaded_sr, duration=duration)
        y = _peak_normalize(y)
        y = np.asarray(y, dtype=np.float32).reshape(-1)
        sf.write(str(destination_path), y, sr)
        return destination_path
    except Exception:
        return None


def convert_noise_to_wav(source, destination, prefix, sr=16000, duration=2.0):
    """Same as convert_to_wav but raises on failure (noise files must be valid)."""
    destination_path = destination / f"{prefix}_{source.stem}.wav"
    suffix = 1
    while destination_path.exists():
        destination_path = destination / f"{prefix}_{suffix}_{source.stem}.wav"
        suffix += 1
    y, loaded_sr = librosa.load(str(source), sr=sr, mono=True)
    if y.size == 0:
        y = np.zeros(int(sr * duration), dtype=np.float32)
    y = select_loudest_window(y, loaded_sr, duration=duration)
    y = _peak_normalize(y)
    y = np.asarray(y, dtype=np.float32).reshape(-1)
    sf.write(str(destination_path), y, sr)
    return destination_path


def extract_mel_spectrogram(audio_path, sr=SR, duration=DURATION):
    """Load a WAV file and convert to normalized Mel-spectrogram tensor."""
    y, _ = librosa.load(audio_path, sr=sr, mono=True)
    if y.size == 0:
        y = np.zeros(int(sr * duration), dtype=np.float32)
    target_len = max(1, int(sr * duration))
    if y.size > target_len:
        y = y[:target_len]

    n_fft = N_FFT
    n_mels = N_MELS
    n_frames = N_FRAMES

    if len(y) < n_fft:
        pad_len = n_fft + (n_frames - 1) - len(y)
        y = np.pad(y, (0, max(0, pad_len)), mode="constant")
        hop_length = 1
    else:
        hop_length = max(1, (len(y) - n_fft) // (n_frames - 1))
        target_samples = n_fft + (n_frames - 1) * hop_length
        if len(y) > target_samples:
            y = y[:target_samples]

    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft,
        hop_length=hop_length, n_mels=n_mels,
        power=2.0, fmax=sr / 2.0
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = np.asarray(mel_db, dtype=np.float32)

    mel_min, mel_max = np.min(mel_db), np.max(mel_db)
    if np.isfinite(mel_min) and np.isfinite(mel_max) and (mel_max - mel_min) > 1e-8:
        mel_db = (mel_db - mel_min) / (mel_max - mel_min + 1e-8)
    else:
        mel_db = np.zeros_like(mel_db, dtype=np.float32)

    mel_db = np.clip(mel_db, 0.0, 1.0)
    if mel_db.shape[1] < n_frames:
        mel_db = np.pad(mel_db, ((0, 0), (0, n_frames - mel_db.shape[1])), mode="constant")
    elif mel_db.shape[1] > n_frames:
        mel_db = mel_db[:, :n_frames]

    return mel_db[np.newaxis, :, :]


def spec_augment(mel_spec, noise_std=0.02, p=0.75):
    """Apply SpecAugment-style data augmentation (noise + gain)."""
    x = np.asarray(mel_spec, dtype=np.float32).copy()
    if x.ndim == 2:
        x = x[np.newaxis, :, :]
    if np.random.rand() < p:
        noise = np.random.normal(0.0, noise_std, size=x.shape).astype(np.float32)
        x = x + noise
        x = np.clip(x, 0.0, 1.0)
    if np.random.rand() < p:
        gain = np.random.uniform(0.8, 1.2)
        x = x * gain
        x = np.clip(x, 0.0, 1.0)
    return x

## 2. Download & Merge Multiple Cry Datasets

In [ ]:
print("=" * 60)
print("  CPDS-AI: Multi-Dataset Acquisition")
print("=" * 60)

# --- Source 1: Menna Ahmed Baby Crying Sounds (Kaggle, ~1313 files) ---
import kagglehub
print("\n[1/3] Downloading Baby Crying Sounds (Menna Ahmed, ~1313 files)...")
cry_dataset_path_1 = Path(kagglehub.dataset_download('mennaahmed23/baby-crying-sounds-dataset'))
print(f"  -> Path: {cry_dataset_path_1}")

# --- Source 2: Donate-a-cry Corpus (GitHub, ~1585 files) ---
print("\n[2/3] Cloning Donate-a-cry corpus (~1585 files)...")
if not Path("donateacry-corpus").is_dir():
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/gveres/donateacry-corpus.git"], check=True)
cry_dataset_path_2 = Path("donateacry-corpus")
print(f"  -> Path: {cry_dataset_path_2}")

# --- Source 3: ESC-50 (Environmental noise + 40 crying_baby samples) ---
print("\n[3/3] Cloning ESC-50 (environment noise + 40 crying_baby samples)...")
if not Path("ESC-50").is_dir():
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/karolpiczak/ESC-50.git"], check=True)
ESC50_ROOT = Path("ESC-50")
metadata_path = next(ESC50_ROOT.rglob("esc50.csv"), None)
audio_root = next((p for p in ESC50_ROOT.rglob("audio") if p.is_dir()), None)

if metadata_path is None or audio_root is None:
    raise FileNotFoundError("ESC-50 must contain meta/esc50.csv and an audio directory.")

# Extract crying_baby files from ESC-50
with metadata_path.open(newline="", encoding="utf-8") as csv_file:
    esc50_records = list(csv.DictReader(csv_file))
esc50_cry_files = [
    audio_root / r["filename"]
    for r in esc50_records
    if r["category"] == "crying_baby" and (audio_root / r["filename"]).is_file()
]
print(f"  -> Extracted {len(esc50_cry_files)} crying_baby files from ESC-50")

print("\nAll sources downloaded successfully!")

## 3. Merge & Filter with AST Teacher Model
We use the MIT Audio Spectrogram Transformer (AST) as a pseudo-labeler to discard low-quality or mislabeled audio files from all sources.

In [ ]:
# Scan all cry audio files from all sources
all_cry_files_raw = []
for source_path in [cry_dataset_path_1, cry_dataset_path_2]:
    found = sorted(p for p in source_path.rglob("*") if p.suffix.lower() in SOURCE_AUDIO_EXTENSIONS)
    print(f"  Found {len(found)} files from {source_path.name}")
    all_cry_files_raw.extend(found)

all_cry_files_raw.extend(esc50_cry_files)
print(f"  Found {len(esc50_cry_files)} files from ESC-50 (crying_baby)")
print(f"\nTotal raw cry files (before filtering): {len(all_cry_files_raw)}")

# Filter using AST Teacher
from transformers import pipeline
print('\nLoading MIT AST model for pseudo-labeling...')
try:
    ast_pipe = pipeline('audio-classification',
                        model='MIT/ast-finetuned-audioset-10-10-0.4593',
                        device=0 if torch.cuda.is_available() else -1)
except Exception as e:
    print(f'  Warning: Failed to load AST model: {e}. Skipping filter step.')
    ast_pipe = None

cry_files = []
if ast_pipe:
    print(f'Filtering {len(all_cry_files_raw)} files with AST Teacher...')
    for i, path in enumerate(all_cry_files_raw):
        if i % 200 == 0:
            print(f"  Progress: {i}/{len(all_cry_files_raw)}")
        try:
            res = ast_pipe(str(path))
            is_cry = any(
                ('cry' in r['label'].lower() or 'infant' in r['label'].lower() or 'baby' in r['label'].lower())
                and r['score'] > 0.1
                for r in res
            )
            if is_cry:
                cry_files.append(path)
        except Exception:
            pass
    print(f'\nAST kept {len(cry_files)} clean samples. Discarded {len(all_cry_files_raw) - len(cry_files)} noisy/invalid files.')
else:
    cry_files = all_cry_files_raw

if not cry_files:
    raise ValueError("No valid cry files found after filtering!")

## 4. Train/Test Split & WAV Conversion

**Key fix:** We balance cry and noise counts, and keep holdout small & balanced.

In [ ]:
# Create output directories
generated_root = Path("/kaggle/working/cpds-audio")
if generated_root.exists():
    shutil.rmtree(generated_root)
base_dir = generated_root / "train"
(base_dir / 'cry').mkdir(parents=True, exist_ok=True)
(base_dir / 'noise').mkdir(parents=True, exist_ok=True)
holdout_dir = generated_root / "holdout_test"
(holdout_dir / "cry").mkdir(parents=True, exist_ok=True)
(holdout_dir / "noise").mkdir(parents=True, exist_ok=True)

# Split cry files: small holdout for final evaluation, rest for training
cry_holdout, cry_train_pool = split_holdout_and_train_pool(cry_files, SEED, EVAL_HOLDOUT_PER_CLASS)
selected_cry = cry_train_pool[:min(MAX_SAMPLES_PER_CLASS, len(cry_train_pool))]

print(f"Cry data allocation:")
print(f"  Train pool: {len(selected_cry)} files")
print(f"  Holdout Test: {len(cry_holdout)} files")

# Convert cry files to standardized WAV (2s, 16kHz, peak-normalized)
print("\nConverting cry files to WAV (2s, 16kHz)...")
converted_cry = [convert_to_wav(f, base_dir / "cry", "cry") for f in selected_cry]
converted_cry = [p for p in converted_cry if p is not None]
converted_cry_holdout = [convert_to_wav(f, holdout_dir / "cry", "cry") for f in cry_holdout]
converted_cry_holdout = [p for p in converted_cry_holdout if p is not None]

# Noise: Select vehicle-relevant categories from ESC-50 (exclude crying_baby)
noise_records = [
    r for r in esc50_records
    if r["category"] in VEHICLE_NOISE_CATEGORIES and (audio_root / r["filename"]).is_file()
]
print(f"\nTotal ESC-50 noise files available: {len(noise_records)}")

# FIX: Use small holdout per-class (like original notebook: 8 per category)
NOISE_HOLDOUT_PER_CAT = 8
noise_holdout_recs, noise_train_pool = split_holdout_and_train_pool(
    noise_records, SEED, NOISE_HOLDOUT_PER_CAT * len(VEHICLE_NOISE_CATEGORIES)
)

# Balance: noise train count matches cry train count (cap by available noise)
selected_noise = noise_train_pool[:min(len(converted_cry), len(noise_train_pool))]

print(f"\nNoise data allocation:")
print(f"  Train: {len(selected_noise)} files")
print(f"  Holdout Test: {len(noise_holdout_recs)} files")

for record in selected_noise:
    convert_noise_to_wav(audio_root / record["filename"], base_dir / "noise", f"esc50_{record['category']}")
for record in noise_holdout_recs:
    convert_noise_to_wav(audio_root / record["filename"], holdout_dir / "noise", f"esc50_{record['category']}")

# Save dataset manifest
manifest = {
    "seed": SEED,
    "cry_sources": [
        "kagglehub: mennaahmed23/baby-crying-sounds-dataset",
        "github: gveres/donateacry-corpus",
        "github: karolpiczak/ESC-50 (crying_baby class)",
    ],
    "esc50_input": "https://github.com/karolpiczak/ESC-50.git",
    "total_raw_cry_files": len(all_cry_files_raw),
    "ast_filtered_cry_files": len(cry_files),
    "cry_train_count": len(converted_cry),
    "cry_holdout_count": len(converted_cry_holdout),
    "noise_train_count": len(selected_noise),
    "noise_holdout_count": len(noise_holdout_recs),
}
(generated_root / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2))

print("\n" + "=" * 60)
print("  Dataset preparation complete!")
print(f"  Train: {len(converted_cry)} cry + {len(selected_noise)} noise")
print(f"  Holdout Test: {len(converted_cry_holdout)} cry + {len(noise_holdout_recs)} noise")
print("=" * 60)

## 5. Model Definition (AudioCNN 4-Block + AudioDataset with SpecAugment)

**Key fixes from v1:**
- 4 conv blocks (16→32→64→128) instead of 3 (matches proven architecture)
- `bias=False` in Conv2d (proper BatchNorm usage)
- SpecAugment data augmentation during training
- 80/20 train/val split from training data (NOT using holdout as val)

In [ ]:
class AudioDataset(Dataset):
    """Loads WAV files, converts to mel-spectrogram tensors with optional augmentation."""
    def __init__(self, file_list, augment=False):
        self.samples = file_list  # list of (path, label_idx)
        self.augment = augment
        if not self.samples:
            raise ValueError("No audio files provided to AudioDataset.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        mel_spec = extract_mel_spectrogram(path)
        if self.augment:
            mel_spec = spec_augment(mel_spec)
        return torch.from_numpy(mel_spec).float(), label


class AudioCNN(nn.Module):
    """4-block CNN for binary audio classification (matches proven architecture)."""
    def __init__(self, num_classes=2, dropout=0.3):
        super(AudioCNN, self).__init__()

        def conv_block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(
            conv_block(1, 16),
            conv_block(16, 32),
            conv_block(32, 64),
            conv_block(64, 128),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        return self.fc(x)

## 6. Training Loop (AdamW + CosineAnnealing + Early Stopping)

**Key fixes:**
- 80/20 train/val split from training data
- AdamW optimizer with weight_decay=1e-4
- CosineAnnealingLR scheduler
- Early stopping only after min_epochs=10

In [ ]:
# Build file lists: (path, label_idx)
def list_class_samples(root, class_names):
    return [(path, index) for index, label in enumerate(class_names)
            for path in (root / label).rglob("*") if path.suffix.lower() in AUDIO_EXTENSIONS]

all_train_samples = list_class_samples(base_dir, CLASS_NAMES)
if not all_train_samples:
    raise ValueError(f"No audio files found under {base_dir}")

# 80/20 train/val split (from training data only, NOT holdout)
shuffled = all_train_samples.copy()
random.Random(SEED).shuffle(shuffled)
split_idx = int(0.8 * len(shuffled))
train_files, val_files = shuffled[:split_idx], shuffled[split_idx:]

train_dataset = AudioDataset(train_files, augment=True)
val_dataset = AudioDataset(val_files, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AudioCNN(num_classes=len(CLASS_NAMES)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()

print(f"Training started")
print(f"  Device: {device}")
print(f"  Train samples: {len(train_dataset)} (80%)")
print(f"  Val samples:   {len(val_dataset)} (20%)")
print(f"  Holdout test:  {len(list_class_samples(holdout_dir, CLASS_NAMES))} (untouched)")
print(f"  Max epochs: {NUM_EPOCHS}, Early stopping patience: {PATIENCE} (min epochs: {EARLY_STOP_MIN_EPOCHS})")
print()

best_val_loss = float('inf')
best_accuracy = -1.0
patience_counter = 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += inputs.size(0)
    scheduler.step()

    # --- Validate ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += inputs.size(0)

    avg_train_loss = train_loss / max(train_total, 1)
    avg_val_loss = val_loss / max(val_total, 1)
    train_acc = train_correct / max(train_total, 1)
    val_acc = val_correct / max(val_total, 1)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"  Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f}")

    # Save best by accuracy
    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save(model.state_dict(), "best_audio_model.pth")

    # Early Stopping (only after min epochs)
    if avg_val_loss < best_val_loss - 1e-4:
        best_val_loss = avg_val_loss
        patience_counter = 0
    else:
        patience_counter += 1
    if (epoch + 1) >= EARLY_STOP_MIN_EPOCHS and patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
        break

# Reload best checkpoint
model.load_state_dict(torch.load("best_audio_model.pth"))
print(f"\nBest model loaded (val accuracy: {best_accuracy:.4f})")

## 7. Evaluation (Curves + Confusion Matrix on HOLDOUT test set)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history["train_loss"], label="Train Loss")
ax1.plot(history["val_loss"], label="Val Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss Curves"); ax1.legend()
ax2.plot(history["train_acc"], label="Train Acc")
ax2.plot(history["val_acc"], label="Val Acc")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.set_title("Accuracy Curves"); ax2.legend()
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Confusion matrix on held-out test set (NEVER seen during training or validation)
holdout_samples = list_class_samples(holdout_dir, CLASS_NAMES)
holdout_dataset = AudioDataset(holdout_samples, augment=False)
holdout_loader = DataLoader(holdout_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in holdout_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy() if hasattr(labels, 'numpy') else labels)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, annot_kws={"size": 16})
plt.title("Held-out Test Confusion Matrix", fontsize=14)
plt.ylabel("True Label", fontsize=12)
plt.xlabel("Predicted Label", fontsize=12)
plt.savefig("holdout_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nClassification Report (Holdout Test Set):")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

## 8. Export ONNX Model

In [ ]:
artifacts_dir = Path("/kaggle/working/artifacts")
artifacts_dir.mkdir(exist_ok=True)

model.eval()
dummy_input = torch.randn(1, 1, N_MELS, N_FRAMES).to(device)
onnx_path = artifacts_dir / "audio_model_fp32.onnx"

torch.onnx.export(
    model, dummy_input, str(onnx_path),
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=17,
)

# Save class labels
labels_path = artifacts_dir / "audio_labels.json"
labels_path.write_text(json.dumps(CLASS_NAMES))

# Copy training artifacts
shutil.copy2("training_curves.png", artifacts_dir / "training_curves.png")
shutil.copy2("holdout_confusion_matrix.png", artifacts_dir / "holdout_confusion_matrix.png")

# Save training history
Path(artifacts_dir / "training_history.json").write_text(json.dumps(history, indent=2))

print(f"ONNX export successful!")
print(f"  Model:  {onnx_path}")
print(f"  Labels: {labels_path}")
print(f"\nNext steps:")
print(f"  1. Download the 'artifacts' folder")
print(f"  2. Rename 'audio_model_fp32.onnx' -> 'audio_model.onnx'")
print(f"  3. Copy to data/models/ in the CPDS-AI project")
print(f"  4. Run: python3 main.py --mode evaluate")